# 03 · SMLP4Rec + destination prior (late fusion)

Thay thế `scripts/late_fusion_destination_prior.py` (branch `exp/late-fusion-destination-prior`). Không huấn luyện lại: mô hình là checkpoint tốt nhất của notebook 01.

```
fusion: score(k) = log p_SMLP4Rec(k | history) + w_p * log p_prior(k | searched destination)
```

- **p_prior**: số đếm cluster theo destination được làm mượt về thị trường (market) của destination, `(n_dest,k + m * p_market,k) / (n_dest + m)`, m = 5; destination chưa thấy → global. Market = `hotel_market` xuất hiện nhiều nhất của destination trong dữ liệu prior, không bao giờ dùng `hotel_market` của chính sự kiện. Số đếm chỉ dùng các booking trước mục tiêu train cuối cùng (trừ 128 s: RecBole lưu thời gian dạng float32), nên không booking valid/test nào đóng góp vào prior của chính nó.
- **Trọng số** `w_p` được chọn trên VALID theo Recall@5 (hòa → chọn trọng số nhỏ hơn), theo từng bucket lịch sử L = 1 | 2-4 | 5-9 | 10+ hoặc một giá trị global; test chỉ đọc một lần.
- **Quy tắc tham chiếu** (tuần 2, không mô hình): các cluster user đã đặt tại destination truy vấn đứng đầu (độ mới 0.7^age), rồi đến prior.
- **Dòng warm** = các mục tiêu RecBole 80/10/10 (L >= 1). **Dòng cold** (booking đầu tiên của mỗi user, L = 0) được chấm riêng chỉ bằng prior, trong cùng các cửa sổ thời gian; "all events" = warm + cold.
- **Metric**: Recall@K và NDCG@K (K = 5, 10, 20); khoảng tin cậy 95% bootstrap theo cặp cho chênh lệch Recall@5 và NDCG@5 trên test.
- **Thay đổi so với script**: các metric phụ của script (MRR@10, MAP@5) bị bỏ.

Bước tiếp theo (notebook 04): thêm tín hiệu lịch sử cùng destination (sameDest) vào fusion này.

Kernel: `smlp4rec`. Cần `data/interim/recbole/expedia_dest/expedia_dest.inter` (`scripts/expedia_to_recbole.py --with-destination`, cùng các dòng và thứ tự với bản export plain) và `train.csv` thô để tính prior.

In [1]:
import functools
import json
import logging
import sys
import time
from pathlib import Path

import numpy as np
import torch  # before pandas (Windows c10.dll issue)

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
torch.load = functools.partial(torch.load, weights_only=False)

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.eval.bootstrap import paired_bootstrap
from nbp.eval.compare import compare_nested
from nbp.eval.metrics import summarize, target_rank
from src.models.smlprec import SMLPREC

torch.autograd.set_detect_anomaly(False)
logging.disable(logging.INFO)

CONFIG = ROOT / "configs" / "smlprec_expedia.yaml"
WORK = ROOT / "data" / "interim" / "recbole"
EVENT_COLS = ROOT / "data" / "interim" / "expedia_event_cols_clean.npz"  # row position = event index = TIME
INTER_DEST = WORK / "expedia_clean_dest" / "expedia_clean_dest.inter"
REF_DIR = ROOT / "results" / "week4_rebuild" / "clean_chain"  # outputs and predecessor results on the collapsed data
OLD_DIR = ROOT / "results" / "week3_implementation"  # same chain on the uncollapsed data (reference only)
REF_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = sorted((ROOT / "experiments").glob("*_expedia_smlp4rec_clean-plain"))[-1]

N_CLUSTERS = 100
M_SMOOTH = 5
EPS = 1e-6
TIME_MARGIN = 0  # TIME is the exact event index: no float32 margin
RECENCY = 0.7  # week-2 same-destination rule (reference)
W_GRID = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0]  # fusion sweep
BUCKETS = [("L=1", 1, 1), ("L=2-4", 2, 4), ("L=5-9", 5, 9), ("L>=10", 10, 10**9)]
KS = (5, 10, 20)
print("run dir:", RUN_DIR)

run dir: C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-08_expedia_smlp4rec_clean-plain


## 1. Dữ liệu kèm destination được tìm

Cùng cách chia với notebook 01, cộng `srch_destination_id` cho mỗi booking, nên mỗi dòng mục tiêu mang destination truy vấn của nó và mỗi vị trí lịch sử mang destination của booking đó.

In [2]:
config = Config(
    model=SMLPREC,
    dataset="expedia_clean_dest",
    config_file_list=[str(CONFIG)],
    config_dict={
        "data_path": str(WORK),
        "checkpoint_dir": str(RUN_DIR),
        "load_col": {"inter": ["user_id", "item_id", "timestamp", "srch_destination_id"]},
    },
)
init_seed(config["seed"], config["reproducibility"])
t0 = time.time()
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
tok2id = dataset.field2token_id[dataset.iid_field]
cluster_ids = np.array([tok2id[str(k)] for k in range(N_CLUSTERS)])  # model column of each raw cluster
n_items = dataset.item_num
ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
ITEM_SEQ_LEN = config["ITEM_LIST_LENGTH_FIELD"]
TIME = config["TIME_FIELD"]


def split_arrays(loader) -> dict:
    """History ids/length/destinations, target, query destination, past-cluster mask of one split."""
    d = loader.dataset.inter_feat
    seq, length = d[ITEM_SEQ].numpy(), d[ITEM_SEQ_LEN].numpy()
    y = d[config["ITEM_ID_FIELD"]].numpy()
    in_hist = np.zeros((len(y), n_items), bool)
    for j in range(seq.shape[1]):
        r = np.flatnonzero(j < length)
        in_hist[r, seq[r, j]] = True
    in_hist[:, 0] = False
    return {
        "seq": seq,
        "seq_t": d[ITEM_SEQ],
        "length": length,
        "length_t": d[ITEM_SEQ_LEN],
        "y": y,
        "old": in_hist[np.arange(len(y)), y],
        "q_dest": d["srch_destination_id"].numpy().astype(np.int64),
        "h_dest": d["srch_destination_id" + config["LIST_SUFFIX"]].numpy().astype(np.int64),
    }


arr = {"valid": split_arrays(valid_data), "test": split_arrays(test_data)}
masks = {sp: {b: (a["length"] >= lo) & (a["length"] <= hi) for b, lo, hi in BUCKETS} for sp, a in arr.items()}
print({sp: len(a["y"]) for sp, a in arr.items()}, f"{time.time() - t0:.0f}s")

{'valid': 213869, 'test': 213869} 56s


## 2. Destination prior (điểm cắt không rò rỉ)

In [3]:
import pandas as pd  # after torch

t_train_max = float(train_data.dataset.inter_feat[TIME].max())
t_valid_max = float(valid_data.dataset.inter_feat[TIME].max())
cut = t_train_max - TIME_MARGIN


def build_prior_tables(cut_event: float) -> dict:
    """Cluster count tables from the bookings strictly before event index cut_event (collapsed data)."""
    z, k = np.load(EVENT_COLS), int(cut_event)
    b = pd.DataFrame({"srch_destination_id": z["dest"][:k], "hotel_market": z["market"][:k], "hotel_cluster": z["item"][:k]}).astype("int64")

    def table(key):
        t = b.groupby([key, "hotel_cluster"]).size().unstack(fill_value=0)
        return t.reindex(columns=range(N_CLUSTERS), fill_value=0)

    dest, mkt = table("srch_destination_id"), table("hotel_market")
    dm = b.groupby(["srch_destination_id", "hotel_market"]).size().reset_index(name="n")
    modal = dm.sort_values("n", ascending=False).drop_duplicates("srch_destination_id")
    glob = np.bincount(b["hotel_cluster"].values, minlength=N_CLUSTERS).astype(np.float64)
    return {
        "n_bookings": len(b),
        "dest": {int(d): r.astype(np.float64) for d, r in zip(dest.index, dest.values)},
        "market": {int(m): r.astype(np.float64) for m, r in zip(mkt.index, mkt.values)},
        "dest2mkt": dict(zip(modal["srch_destination_id"].astype(int), modal["hotel_market"].astype(int))),
        "global": glob / glob.sum(),
    }


def prior_vector(d: int) -> np.ndarray:
    """p(cluster | destination), shape (100,), raw cluster order."""
    row = tables["dest"].get(d)
    if row is None:
        return tables["global"]
    pm = tables["market"][tables["dest2mkt"][d]]
    pm = pm / pm.sum()
    return (row + M_SMOOTH * pm) / (row.sum() + M_SMOOTH)


def prior_logp(dests: np.ndarray) -> np.ndarray:
    """(n, n_items) log prior, columns = model item ids; padding column 0 stays 0 (masked at ranking)."""
    uniq, inv = np.unique(dests, return_inverse=True)
    tab = np.stack([np.log(prior_vector(int(d)) + EPS) for d in uniq]).astype(np.float32)
    out = np.zeros((len(dests), n_items), np.float32)
    out[:, cluster_ids] = tab[inv]
    return out


t1 = time.time()
tables = build_prior_tables(cut)
print(f"prior cut (unix s) {cut:.0f}; {tables['n_bookings']} bookings, {len(tables['dest'])} destinations; {time.time() - t1:.0f}s")

prior cut (unix s) 2424705; 2424705 bookings, 34507 destinations; 1s


## 3. Các thành phần điểm theo từng tập chia

- `ml`: log-softmax của điểm mô hình (cột padding -inf).
- `pr`: log prior của destination truy vấn.

`rank_rows` đặt cột padding thành -inf trên mảng được đưa vào (luôn là ma trận điểm mới dựng) và trả về thứ hạng của cluster đúng.

In [4]:
model = SMLPREC(config, dataset)
model.load_state_dict(torch.load(RUN_DIR / "best.pth")["state_dict"])


@torch.no_grad()
def model_logp(a: dict, batch: int = 8192) -> np.ndarray:
    model.eval()
    out = []
    for i in range(0, len(a["y"]), batch):
        s = model.full_sort_predict(
            {model.ITEM_SEQ: a["seq_t"][i : i + batch], model.ITEM_SEQ_LEN: a["length_t"][i : i + batch]}
        )
        s[:, 0] = float("-inf")
        out.append(torch.log_softmax(s, dim=1).numpy())
    return np.concatenate(out)


def same_dest_rule(a: dict, prior: np.ndarray) -> np.ndarray:
    """No-learning rule: past clusters at the query destination first (recency 0.7^age), then the prior."""
    s = prior.copy()
    for j in range(a["seq"].shape[1]):
        idx = np.flatnonzero((j < a["length"]) & (a["h_dest"][:, j] == a["q_dest"]))
        bonus = 1000.0 * RECENCY ** (a["length"][idx] - 1 - j)
        cl = a["seq"][idx, j]
        s[idx, cl] = np.maximum(s[idx, cl], prior[idx, cl] + bonus)
    return s


def rank_rows(scores: np.ndarray, y: np.ndarray) -> np.ndarray:
    scores[:, 0] = -np.inf
    return target_rank(scores, y)


def fused(ml, pr, w) -> np.ndarray:
    """ml + w * pr; w scalar or per-row (n,)."""
    w = np.asarray(w, np.float32)
    return ml + (w[:, None] if w.ndim else w) * pr


ml, pr = {}, {}
for sp, a in arr.items():
    ml[sp], pr[sp] = model_logp(a), prior_logp(a["q_dest"])

ref01 = json.loads((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))
for sp in arr:
    got = summarize(rank_rows(fused(ml[sp], pr[sp], 0.0), arr[sp]["y"]))["recall@5"]
    print(f"sanity {sp}: plain Recall@5 {got} vs notebook 01 best epoch {ref01[sp]['recall@5']}")

sanity valid: plain Recall@5 0.3204 vs notebook 01 best epoch 0.3204
sanity test: plain Recall@5 0.3212 vs notebook 01 best epoch 0.3212


## 4. Late fusion: quét w_p, chọn trên valid

In [5]:
sweep = {sp: {} for sp in arr}
for w in W_GRID:
    for sp, a in arr.items():
        rk = rank_rows(fused(ml[sp], pr[sp], w), a["y"])
        sweep[sp][str(w)] = {"all": summarize(rk), **{b: summarize(rk, m) for b, m in masks[sp].items()}}

chosen_w = {b: max(W_GRID, key=lambda w: (sweep["valid"][str(w)][b]["recall@5"], -w)) for b, _, _ in BUCKETS}
best_global_w = max(W_GRID, key=lambda w: (sweep["valid"][str(w)]["all"]["recall@5"], -w))
print("chosen w_p per bucket:", chosen_w, "| best global w_p:", best_global_w)


def per_row_w(sp: str) -> np.ndarray:
    w = np.zeros(len(arr[sp]["y"]), np.float32)
    for b, m in masks[sp].items():
        w[m] = chosen_w[b]
    return w


fusion_variants, fusion_ranks = {}, {}
for sp, a in arr.items():
    variants = {
        "plain_model (w=0)": fused(ml[sp], pr[sp], 0.0),
        "destination_prior_only": pr[sp] + 1e-6 * np.nan_to_num(ml[sp], neginf=0.0),
        "same_dest_history_then_prior": same_dest_rule(a, pr[sp]),
        f"fusion_global_w={best_global_w}": fused(ml[sp], pr[sp], best_global_w),
        "fusion_per_bucket_w": fused(ml[sp], pr[sp], per_row_w(sp)),
    }
    fusion_variants[sp], fusion_ranks[sp] = {}, {}
    for name, sc in variants.items():
        rk = rank_rows(sc, a["y"])
        fusion_ranks[sp][name] = rk
        fusion_variants[sp][name] = {
            "all": summarize(rk),
            "old_target_rows": summarize(rk, a["old"]),
            "new_target_rows": summarize(rk, ~a["old"]),
            **{b: summarize(rk, m) for b, m in masks[sp].items()},
        }
pd.DataFrame({k: v["all"] for k, v in fusion_variants["test"].items()}).T

chosen w_p per bucket: {'L=1': 2.0, 'L=2-4': 1.5, 'L=5-9': 1.5, 'L>=10': 1.5} | best global w_p: 1.5


,n,recall@5,recall@10,recall@20,ndcg@5,ndcg@10,ndcg@20
plain_model (w=0),213869.0,0.3212,0.4530,0.6230,0.2274,0.2698,0.3127
destination_prior_only,213869.0,0.5294,0.7207,0.8842,0.3627,0.4246,0.4661
same_dest_history_then_prior,213869.0,0.5798,0.7490,0.8948,0.4251,0.4798,0.5169
fusion_global_w=1.5,213869.0,0.5851,0.7617,0.9073,0.4139,0.4712,0.5083
fusion_per_bucket_w,213869.0,0.5847,0.7616,0.9072,0.4136,0.4710,0.5080


## 5. User cold (L = 0): chỉ dùng prior

Booking đầu tiên của mỗi user, cửa sổ valid = [mục tiêu train cuối, mục tiêu valid cuối), cửa sổ test = từ mục tiêu valid cuối trở đi.

In [6]:
inter = pd.read_csv(INTER_DEST, sep="\t").sort_values(["user_id:token", "timestamp:float"], kind="stable")
first = inter.drop_duplicates("user_id:token", keep="first")
windows = {
    "valid": (first["timestamp:float"] >= t_train_max) & (first["timestamp:float"] < t_valid_max),
    "test": first["timestamp:float"] >= t_valid_max,
}
glob_row = np.full(n_items, -np.inf, np.float32)
glob_row[cluster_ids] = np.log(tables["global"] + EPS)

cold_rank, cold = {}, {}
for sp, mask in windows.items():
    f = first[mask]
    y = np.array([tok2id[str(int(c))] for c in f["item_id:token"]])
    dests = f["srch_destination_id:float"].values.astype(np.int64)
    cold_rank[sp] = {
        "prior": rank_rows(prior_logp(dests), y),
        "global": rank_rows(np.tile(glob_row, (len(y), 1)), y),
    }
    n_dest = np.array([tables["dest"][d].sum() if d in tables["dest"] else 0 for d in dests])
    support = {"n_dest=0": n_dest == 0, "n_dest 1-19": (n_dest >= 1) & (n_dest < 20), "n_dest>=20": n_dest >= 20}
    cold[sp] = {
        "n_cold_rows": len(y),
        "prior_only": summarize(cold_rank[sp]["prior"]),
        "global_popularity": summarize(cold_rank[sp]["global"]),
        "prior_only_by_destination_support": {
            k: {**summarize(cold_rank[sp]["prior"], m), "share": round(float(m.mean()), 4)} for k, m in support.items()
        },
    }
    print(sp, "cold rows", len(y), "| prior-only", cold[sp]["prior_only"], "\n   global", cold[sp]["global_popularity"])

combined = {}
for sp in arr:
    warm, n_warm = fusion_ranks[sp], len(arr[sp]["y"])
    cp, cg = cold_rank[sp]["prior"], cold_rank[sp]["global"]
    combined[sp] = {
        "n_warm_rows": n_warm,
        "n_cold_rows": len(cp),
        "cold_share_of_events": round(len(cp) / (len(cp) + n_warm), 4),
        "hybrid: fusion on warm + prior-only on cold": summarize(np.concatenate([warm[f"fusion_global_w={best_global_w}"], cp])),
        "prior-only for everyone": summarize(np.concatenate([warm["destination_prior_only"], cp])),
        "week-2 heuristic (same-dest history, prior when cold)": summarize(np.concatenate([warm["same_dest_history_then_prior"], cp])),
        "plain SMLP4Rec on warm + global popularity on cold": summarize(np.concatenate([warm["plain_model (w=0)"], cg])),
    }
pd.DataFrame({k: v for k, v in combined["test"].items() if isinstance(v, dict)}).T

valid cold rows 50769 | prior-only {'n': 50769, 'recall@5': 0.5295, 'recall@10': 0.716, 'recall@20': 0.8849, 'ndcg@5': 0.3619, 'ndcg@10': 0.4222, 'ndcg@20': 0.4651} 
   global {'n': 50769, 'recall@5': 0.125, 'recall@10': 0.21, 'recall@20': 0.3651, 'ndcg@5': 0.0799, 'ndcg@10': 0.1072, 'ndcg@20': 0.146}


test cold rows 49462 | prior-only {'n': 49462, 'recall@5': 0.5129, 'recall@10': 0.7015, 'recall@20': 0.8757, 'ndcg@5': 0.3511, 'ndcg@10': 0.4121, 'ndcg@20': 0.4564} 
   global {'n': 49462, 'recall@5': 0.1318, 'recall@10': 0.227, 'recall@20': 0.3635, 'ndcg@5': 0.0814, 'ndcg@10': 0.1119, 'ndcg@20': 0.1459}


,n,recall@5,recall@10,recall@20,ndcg@5,ndcg@10,ndcg@20
hybrid: fusion on warm + prior-only on cold,263331.0,0.5715,0.7504,0.9014,0.4021,0.4601,0.4985
prior-only for everyone,263331.0,0.5263,0.7171,0.8826,0.3605,0.4222,0.4643
"week-2 heuristic (same-dest history, prior when cold)",263331.0,0.5672,0.7401,0.8912,0.4112,0.4671,0.5055
plain SMLP4Rec on warm + global popularity on cold,263331.0,0.2856,0.4105,0.5743,0.2000,0.2401,0.2813


## 6. Bootstrap theo cặp trên test (1000 lần lấy mẫu lại, seed 0)

In [7]:
FKEY = f"fusion_global_w={best_global_w}"
COMPARISONS = [
    (FKEY, "destination_prior_only"),
    (FKEY, "same_dest_history_then_prior"),
    (FKEY, "plain_model (w=0)"),
]


def cold_for(key: str) -> np.ndarray:
    """Cold rows: global popularity for the plain model (no cold handling), prior otherwise."""
    return cold_rank["test"]["global" if key.startswith("plain") else "prior"]


boot = {}
for ka, kb in COMPARISONS:
    ra, rb = fusion_ranks["test"][ka], fusion_ranks["test"][kb]
    boot[f"{ka} minus {kb}"] = {
        "warm": paired_bootstrap(ra, rb, k=5),
        "all_events": paired_bootstrap(
            np.concatenate([ra, cold_for(ka)]), np.concatenate([rb, cold_for(kb)]), k=5
        ),
    }
pd.DataFrame(
    {
        (k, scope): {f"{m} diff": v[scope][m]["diff"] for m in ("recall@5", "ndcg@5")}
        | {f"{m} ci95": v[scope][m]["ci95"] for m in ("recall@5", "ndcg@5")}
        for k, v in boot.items()
        for scope in ("warm", "all_events")
    }
).T

recall@5 diff  \
fusion_global_w=1.5 minus destination_prior_only   warm              0.0556   
                                                   all_events        0.0452   
fusion_global_w=1.5 minus same_dest_history_the... warm              0.0053   
                                                   all_events        0.0043   
fusion_global_w=1.5 minus plain_model (w=0)        warm              0.2639   
                                                   all_events        0.2859   

                                                              ndcg@5 diff  \
fusion_global_w=1.5 minus destination_prior_only   warm            0.0512   
                                                   all_events      0.0416   
fusion_global_w=1.5 minus same_dest_history_the... warm           -0.0112   
                                                   all_events     -0.0091   
fusion_global_w=1.5 minus plain_model (w=0)        warm            0.1865   
                                                   all_events      0.2021   

                                                                  recall@5 ci95  \
fusion_global_w=1.5 minus destination_prior_only   warm         [0.054, 0.0573]   
                                                   all_events  [0.0438, 0.0464]   
fusion_global_w=1.5 minus same_dest_history_the... warm        [0.0037, 0.0069]   
                                                   all_events   [0.003, 0.0055]   
fusion_global_w=1.5 minus plain_model (w=0)        warm        [0.2616, 0.2661]   
                                                   all_events   [0.2838, 0.288]   

                                                                    ndcg@5 ci95  
fusion_global_w=1.5 minus destination_prior_only   warm        [0.0501, 0.0524]  
                                                   all_events  [0.0406, 0.0425]  
fusion_global_w=1.5 minus same_dest_history_the... warm        [-0.0123, -0.01]  
                                                   all_events  [-0.01, -0.0081]  
fusion_global_w=1.5 minus plain_model (w=0)        warm         [0.1848, 0.188]  
                                                   all_events  [0.2007, 0.2036]

## 7. Lưu và so sánh với lượt chạy của script

Kết quả được định dạng lại theo bố cục JSON của script; mọi giá trị số chung đều được so sánh (Recall@K, NDCG@10, trọng số đã chọn, số dòng). Các giá trị MRR / MAP của script cố ý không có tương ứng ở đây. Sau khi so sánh, kết quả của notebook (kèm NDCG@5 và bootstrap) thay thế JSON đã lưu, mà notebook 04 và deck tuần 3 đọc.

In [8]:
fusion_result = {
    "prior": {"smoothing_m": M_SMOOTH, "eps": EPS, "cutoff_unix_seconds": cut, "n_bookings": tables["n_bookings"], "n_destinations": len(tables["dest"])},
    "w_grid": W_GRID,
    "chosen_w_per_bucket_on_valid_recall@5": chosen_w,
    "best_global_w_on_valid": best_global_w,
    "rows": {sp: len(arr[sp]["y"]) for sp in arr},
    "share_of_rows_per_bucket": {sp: {b: round(float(m.mean()), 4) for b, m in masks[sp].items()} for sp in arr},
    "variants": fusion_variants,
    "cold_users_prior_only": cold,
    "combined_warm_plus_cold": combined,
    "sweep": sweep,
    "bootstrap_test_95ci": boot,
}
(RUN_DIR / "late_fusion.json").write_text(json.dumps(fusion_result, indent=2), encoding="utf-8")

# INFORMATIONAL: compare with the same notebook on the uncollapsed data (OLD_DIR); the new result goes to REF_DIR
ref = json.loads((OLD_DIR / "smlprec_expedia_late_fusion.json").read_text(encoding="utf-8"))
cmp = pd.DataFrame(compare_nested(ref, fusion_result), columns=["path", "script", "notebook", "diff"])
pd.set_option("display.max_rows", 300)
print(f"compared {len(cmp)} values; identical: {(cmp['diff'].abs() < 1e-9).sum()}; max |diff|: {cmp['diff'].abs().max():.6f}")
(REF_DIR / "smlprec_expedia_late_fusion.json").write_text(json.dumps(fusion_result, indent=2), encoding="utf-8")
print("saved", REF_DIR / "smlprec_expedia_late_fusion.json")
cmp[cmp["diff"].abs() >= 1e-9] if (cmp["diff"].abs() >= 1e-9).any() else cmp.head(20)

compared 1395 values; identical: 48; max |diff|: 38377.000000
saved C:\workspace\PROJECTS\next-best-product-recommendation\results\week4_rebuild\clean_chain\smlprec_expedia_late_fusion.json


,path,script,notebook,diff
2,prior.n_bookings,2.463082e+06,2.424705e+06,-38377.0000
3,prior.n_destinations,3.450600e+04,3.450700e+04,1.0000
19,rows.valid,2.186700e+05,2.138690e+05,-4801.0000
20,rows.test,2.186700e+05,2.138690e+05,-4801.0000
21,share_of_rows_per_bucket.valid.L=1,1.783000e-01,1.808000e-01,0.0025
...,...,...,...,...
1390,bootstrap_test_95ci.fusion_global_w=1.5 minus ...,2.761000e-01,2.838000e-01,0.0077
1391,bootstrap_test_95ci.fusion_global_w=1.5 minus ...,2.802000e-01,2.880000e-01,0.0078
1392,bootstrap_test_95ci.fusion_global_w=1.5 minus ...,1.955000e-01,2.021000e-01,0.0066
1393,bootstrap_test_95ci.fusion_global_w=1.5 minus ...,1.939000e-01,2.007000e-01,0.0068
